In [2]:
import os
import json
import numpy as np
import soundfile as sf
import pandas as pd
from sklearn.model_selection import train_test_split

# --- Configuration & File Paths ---
MANIFEST_FILE = "master_events_clean.txt"
TRAIN_JSON = "temporal_train_hybrid.json"
VAL_JSON = "temporal_val_hybrid.json"
TEST_JSON = "temporal_test_hybrid.json"
FEATURE_NAMES_JSON = "hybrid_feature_names.json"

VOLTS_PER_STEP = 1.90101491444e-07
AMPS_PER_STEP = 4.9224284384e-08
ADC_MAX = 2**31
SAMPLING_RATE = 16000
CYCLE_SAMPLES = 320  # one 50Hz cycle

TIME_STEPS_SEC = [0, 3, 8, 15, 30, 60]
CLASSES = ["Kettle", "Fridge", "WashingMachine", "Microwave", "Dishwasher"]


def find_suffixed_flac(folder, prefix):
    if not os.path.exists(folder):
        return None
    for f in os.listdir(folder):
        if f.startswith(prefix) and f.endswith(".flac"):
            return os.path.join(folder, f)
    return None


def read_baseline_cycle(flac_path, base_active_start):
    quiet_start = base_active_start - (5 * SAMPLING_RATE)
    if quiet_start < 0:
        return None
    try:
        data_quiet, _ = sf.read(flac_path, start=quiet_start, stop=quiet_start + CYCLE_SAMPLES)
        if len(data_quiet) < CYCLE_SAMPLES:
            return None
        return data_quiet
    except Exception:
        return None


def extract_onset_features(flac_path, base_active_start, offset_sec, quiet_baseline_cycle):
    sample_offset = int(offset_sec * SAMPLING_RATE)
    active_start = base_active_start + sample_offset
    if active_start < 0:
        return None
    try:
        data_active, _ = sf.read(flac_path, start=active_start, stop=active_start + CYCLE_SAMPLES)
        if len(data_active) < CYCLE_SAMPLES:
            return None

        v = data_active[:, 0] * ADC_MAX * VOLTS_PER_STEP
        i_delta = (data_active[:, 1] - quiet_baseline_cycle[:, 1]) * ADC_MAX * AMPS_PER_STEP

        v_rms = np.sqrt(np.mean(v**2))
        i_rms = np.sqrt(np.mean(i_delta**2))
        p = np.mean(v * i_delta)
        apparent = v_rms * i_rms
        q = np.sqrt(max(0.0, apparent**2 - p**2))
        power_factor = float(abs(p) / (apparent + 1e-6))

        i_peak = np.max(np.abs(i_delta))
        crest_factor = float(i_peak / (i_rms + 1e-6))

        fft_vals = np.fft.rfft(i_delta)
        fundamental_mag = np.abs(fft_vals[1]) if len(fft_vals) > 1 else 1e-6
        h3 = float(np.abs(fft_vals[3]) / fundamental_mag) if len(fft_vals) > 3 else 0.0
        h5 = float(np.abs(fft_vals[5]) / fundamental_mag) if len(fft_vals) > 5 else 0.0
        h7 = float(np.abs(fft_vals[7]) / fundamental_mag) if len(fft_vals) > 7 else 0.0

        return {
            "p": float(p), "q": float(q), "irms": float(i_rms),
            "crest_factor": crest_factor, "power_factor": power_factor,
            "h3": h3, "h5": h5, "h7": h7,
        }
    except Exception:
        return None


def onset_avg(steps, key, n=2):
    return float(np.mean([s[key] for s in steps[:n]]))


# Build Feature Name List dynamically
FEATURE_NAMES = ["p_max", "p_delta", "p_std", "crest_factor", "power_factor", "h3", "h5", "h7"]
for t in TIME_STEPS_SEC:
    FEATURE_NAMES += [f"p_t{t}", f"irms_t{t}"]


def build_feature_vector(steps):
    p_vals = [s["p"] for s in steps]
    p_arr = np.array(p_vals)
    p_max = float(p_arr.max())
    nonzero = p_arr[p_arr > 10]
    p_min = float(nonzero.min()) if len(nonzero) else 0.0

    vec = [
        p_max,
        p_max - p_min,
        float(p_arr.std()),
        onset_avg(steps, "crest_factor"),
        onset_avg(steps, "power_factor"),
        onset_avg(steps, "h3"),
        onset_avg(steps, "h5"),
        onset_avg(steps, "h7"),
    ]
    for idx, t in enumerate(TIME_STEPS_SEC):
        vec.append(steps[idx]["p"])
        vec.append(steps[idx]["irms"])
    return vec


def compile_sequence_prompt(steps):
    p_vals = [s["p"] for s in steps]
    i_vals = [s["irms"] for s in steps]
    timeline_lines = [f"- t={t}s: P={p_vals[idx]:.1f}W, Irms={i_vals[idx]:.3f}A" for idx, t in enumerate(TIME_STEPS_SEC)]
    profile_sequence = "\n".join(timeline_lines)
    prompt_text = (
        f"Analyze this electrical event onset:\n{profile_sequence}\n\n"
        f"Identify the household appliance operating during this sequence. "
        f"Options: Kettle, Fridge, WashingMachine, Microwave, Dishwasher. Answer with the class name only."
    )
    return prompt_text


# --- Main Pipeline Execution ---
if __name__ == "__main__":
    print("Extracting features for all events (Unaugmented/Hybrid format)...")
    df = pd.read_csv(MANIFEST_FILE)

    raw_rows = []
    skipped = 0
    for idx, row in df.iterrows():
        appliance = row['Appliance']
        ts = int(row['Timestamp'])
        year = row['Year']
        week = row['Week']
        prefix = row['File_Prefix']

        local_week_folder = os.path.join(str(year), str(week))
        flac_path = find_suffixed_flac(local_week_folder, prefix)
        if not flac_path:
            continue

        hour_ts = (ts // 3600) * 3600
        base_active_start = int((ts - hour_ts) * SAMPLING_RATE)

        quiet_cycle = read_baseline_cycle(flac_path, base_active_start)
        if quiet_cycle is None:
            skipped += 1
            continue

        steps = []
        valid = True
        for step in TIME_STEPS_SEC:
            feats = extract_onset_features(flac_path, base_active_start, step, quiet_cycle)
            if feats is None:
                valid = False
                break
            steps.append(feats)
        if not valid:
            skipped += 1
            continue

        raw_rows.append({"appliance": appliance, "steps": steps})

    df_raw = pd.DataFrame(raw_rows)
    print(f"Extracted {len(df_raw)} unique event sequences, skipped {skipped} incomplete windows.")

    # Pass 2: Build hybrid records containing text, label, and exact numeric floats
    all_samples = []
    for _, r in df_raw.iterrows():
        all_samples.append({
            "text": compile_sequence_prompt(r["steps"]),
            "label": r["appliance"],
            "numeric_features": build_feature_vector(r["steps"]),
        })

    df_samples = pd.DataFrame(all_samples)
    print(f"Built {len(df_samples)} hybrid records containing {len(FEATURE_NAMES)} raw numeric floats.")

    # --- Stratified Clean Dataset Splitting ---
    # Since there are no shifted duplicates, a direct stratified split is safe and leakage-free

    
    train_df, temp_df = train_test_split(
        df_samples, 
        train_size=0.30,  # Explicitly forces exactly 30% into your training group
        stratify=df_samples['label'], 
        random_state=42
    )


    val_df, test_df = train_test_split(
        temp_df, 
        train_size=30/70,  # Automatically calculates the exact ratio required
        stratify=temp_df['label'], 
        random_state=42
    )

    # Save clean datasets to disk
    with open(TRAIN_JSON, "w") as f:
        json.dump(train_df[['text', 'label', 'numeric_features']].to_dict(orient='records'), f)
    with open(VAL_JSON, "w") as f:
        json.dump(val_df[['text', 'label', 'numeric_features']].to_dict(orient='records'), f)
    with open(TEST_JSON, "w") as f:
        json.dump(test_df[['text', 'label', 'numeric_features']].to_dict(orient='records'), f)
    with open(FEATURE_NAMES_JSON, "w") as f:
        json.dump(FEATURE_NAMES, f)

    print("\n--- Pipeline Execution Completed Cleanly ---")
    print(f"  Train Set: {len(train_df)}  | class counts: {train_df['label'].value_counts().to_dict()}")
    print(f"  Val Set:   {len(val_df)}   | class counts: {val_df['label'].value_counts().to_dict()}")
    print(f"  Test Set:  {len(test_df)}  | class counts: {test_df['label'].value_counts().to_dict()}")
    print(f"\nSaved files: {TRAIN_JSON}, {VAL_JSON}, {TEST_JSON}, {FEATURE_NAMES_JSON}")

Extracting features for all events (Unaugmented/Hybrid format)...
Extracted 2220 unique event sequences, skipped 242 incomplete windows.
Built 2220 hybrid records containing 20 raw numeric floats.

--- Pipeline Execution Completed Cleanly ---
  Train Set: 666  | class counts: {'WashingMachine': 146, 'Dishwasher': 144, 'Microwave': 140, 'Kettle': 132, 'Fridge': 104}
  Val Set:   666   | class counts: {'WashingMachine': 146, 'Dishwasher': 144, 'Microwave': 140, 'Kettle': 133, 'Fridge': 103}
  Test Set:  888  | class counts: {'WashingMachine': 194, 'Dishwasher': 193, 'Microwave': 187, 'Kettle': 176, 'Fridge': 138}

Saved files: temporal_train_hybrid.json, temporal_val_hybrid.json, temporal_test_hybrid.json, hybrid_feature_names.json
